# exp_008 — QLoRA Fine-tune of Qwen3-4B

**Runtime:** Colab Pro — A100 40GB  
**Goal:** Fine-tune Qwen3-4B on NuminaMath-CoT + public correct responses, merge weights, push to HuggingFace.  
**Time estimate:** ~1.5–2 hrs for 2 epochs on 25K examples.

### Before running
1. Set `HF_TOKEN` and `HF_REPO_ID` in Cell 2.
2. Upload `public.jsonl`, `public_responses.jsonl`, and `results.jsonl` from exp_004 (see Cell 6 instructions).
3. Runtime → Change runtime type → A100 GPU.

In [ ]:
# Cell 1 — Install dependencies
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q trl datasets bitsandbytes

In [ ]:
# Cell 2 — Config
# Add your HF token via Colab Secrets (left sidebar → key icon → add HF_TOKEN)
from google.colab import userdata
HF_TOKEN = userdata.get("HF_TOKEN")

HF_REPO_ID   = "TrevorDuong/qwen3-4b-math-qlora"  # will be created as private

MODEL_NAME          = "Qwen/Qwen3-4B"
MAX_SEQ_LENGTH      = 1024   # halving from 2048; attention is O(n²) so this is a big speedup
NUMINAMATH_N        = 5000   # fits compute budget (~25-75 min on A100)
LORA_R              = 16
LORA_ALPHA          = 32
TRAIN_BATCH_SIZE    = 8      # batch=16 OOM'd at seq=1024 with grad_checkpointing=False on 80GB
GRAD_ACCUM_STEPS    = 2      # effective batch = 16
NUM_EPOCHS          = 2      # 5K × 2 epochs gives LoRA enough steps to converge on format
LEARNING_RATE       = 2e-4
RANDOM_SEED         = 42

In [ ]:
# Cell 3 — Load model in 4-bit + apply QLoRA
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,           # auto-detect (bfloat16 on A100)
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=LORA_ALPHA,
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing=False,   # 80GB A100 has headroom — no need to pay the ~35% throughput tax
    random_state=RANDOM_SEED,
)
print(model.print_trainable_parameters())

In [ ]:
# Cell 4 — System prompts (exp_004 config — best found so far)
SYSTEM_PROMPT_MATH = (
    "You are an expert mathematician. Solve the problem step-by-step. "
    "Put your final answer inside \\boxed{}. "
    "If the problem has multiple sub-answers, separate them by commas inside a single \\boxed{}, "
    "e.g. \\boxed{3, 7}."
)

SYSTEM_PROMPT_MCQ = (
    "You are an expert mathematician. "
    "Read the problem and the answer choices below, then select the single best answer. "
    "Output ONLY the letter of your chosen option inside \\boxed{}, e.g. \\boxed{C}."
)

In [ ]:
# Cell 5 — Data formatting helpers
import re

def extract_boxed_and_reasoning(text):
    """Return (reasoning_text, boxed_expression) by splitting at the last \\boxed{."""
    idx = text.rfind(r'\boxed{')
    if idx == -1:
        return text.strip(), ""
    # Walk forward to find matching closing brace.
    # Only check depth==0 AFTER seeing the opening '{' of \boxed{,
    # otherwise the loop terminates on the very first character (the backslash).
    depth, end = 0, idx
    seen_open = False
    for i, ch in enumerate(text[idx:]):
        if ch == '{':
            depth += 1
            seen_open = True
        elif ch == '}':
            depth -= 1
        if seen_open and depth == 0:
            end = idx + i + 1
            break
    boxed = text[idx:end]
    reasoning = text[:idx].strip()
    return reasoning, boxed


def make_messages(system, question, reasoning, boxed):
    """Build the ChatML messages list for one training example."""
    assistant = f"<think>\n{reasoning}\n</think>\n\n{boxed}"
    return [
        {"role": "system",    "content": system},
        {"role": "user",      "content": question},
        {"role": "assistant", "content": assistant},
    ]


def apply_chat_template(messages):
    """Render messages to a single string using the tokenizer's template."""
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )


# Sanity check: helper actually returns a complete \boxed{...}
_test = r"some reasoning. \boxed{42}"
_r, _b = extract_boxed_and_reasoning(_test)
assert _b == r"\boxed{42}", f"extract_boxed broken: got {_b!r}"
_test2 = r"reasoning \boxed{\frac{1}{2}} done"
_r2, _b2 = extract_boxed_and_reasoning(_test2)
assert _b2 == r"\boxed{\frac{1}{2}}", f"nested braces broken: got {_b2!r}"
print("Helpers defined and self-tested.")

In [ ]:
# Cell 6 — Load NuminaMath-CoT
from datasets import load_dataset
import random

random.seed(RANDOM_SEED)

# Skip re-download if already cached in this kernel
if "raw" not in globals():
    print("Downloading NuminaMath-CoT (this may take a few minutes)...")
    raw = load_dataset("AI-MO/NuminaMath-CoT", split="train")
print(f"Total rows: {len(raw)}")

# Filter: must have \boxed in solution, reasonable lengths
def is_valid_numinamath(ex):
    sol = ex.get("solution", "")
    prob = ex.get("problem", "")
    return (
        r"\boxed" in sol
        and 20 < len(prob) < 2000
        and 50 < len(sol) < 6000
    )

if "filtered" not in globals():
    filtered = raw.filter(is_valid_numinamath, num_proc=4)
print(f"After filtering: {len(filtered)}")

# Random sample
n = min(NUMINAMATH_N, len(filtered))
indices = random.sample(range(len(filtered)), n)
sample = filtered.select(indices)
print(f"Sampled: {len(sample)}")

# Format
numinamath_texts = []
skipped = 0
for ex in sample:
    reasoning, boxed = extract_boxed_and_reasoning(ex["solution"])
    # Drop anything where the helper failed to find a real \boxed{...}
    if not boxed or boxed == "\\" or "{" not in boxed:
        skipped += 1
        continue
    msgs = make_messages(SYSTEM_PROMPT_MATH, ex["problem"], reasoning, boxed)
    numinamath_texts.append(apply_chat_template(msgs))

print(f"NuminaMath formatted: {len(numinamath_texts)}  (skipped {skipped} with no valid \\boxed)")

In [ ]:
# Cell 7 — Load public correct responses (exp_004)
# Uploads files one at a time since they come from different local directories.
# Skips upload prompts if files are already present (so re-running this cell after
# fixing a bug doesn't make you re-upload everything).

import json, os
from google.colab import files

PUBLIC_JSONL_PATH       = "/content/public.jsonl"
PUBLIC_RESPONSES_PATH   = "/content/public_responses.jsonl"
RESULTS_PATH            = "/content/results.jsonl"

if not os.path.exists(PUBLIC_JSONL_PATH):
    print("Upload 1/3: data/public.jsonl")
    files.upload()
else:
    print(f"Already present: {PUBLIC_JSONL_PATH}")

if not os.path.exists(PUBLIC_RESPONSES_PATH):
    print("Upload 2/3: experiments/exp_004_fewshot_prompts/public_responses.jsonl")
    files.upload()
else:
    print(f"Already present: {PUBLIC_RESPONSES_PATH}")

if not os.path.exists(RESULTS_PATH):
    print("Upload 3/3: experiments/exp_004_fewshot_prompts/results.jsonl")
    files.upload()
else:
    print(f"Already present: {RESULTS_PATH}")

# Load ground-truth questions keyed by id
questions = {}
with open(PUBLIC_JSONL_PATH) as f:
    for line in f:
        ex = json.loads(line)
        questions[ex["id"]] = ex
print(f"Questions loaded: {len(questions)}")

# Load correct IDs from scored results
correct_ids = set()
with open(RESULTS_PATH) as f:
    for line in f:
        r = json.loads(line)
        if r.get("correct"):
            correct_ids.add(r["id"])
print(f"Correct IDs: {len(correct_ids)}")

# options is a list — map index to letter A, B, C...
LETTERS = "ABCDEFGHIJ"

# Format correct responses
public_texts = []
skipped = 0
with open(PUBLIC_RESPONSES_PATH) as f:
    for line in f:
        r = json.loads(line)
        if r["id"] not in correct_ids:
            continue
        q = questions[r["id"]]
        system = SYSTEM_PROMPT_MCQ if r.get("is_mcq") else SYSTEM_PROMPT_MATH

        question_text = q["question"]
        if r.get("is_mcq") and "options" in q:
            opts = "\n".join(f"{LETTERS[i]}. {v}" for i, v in enumerate(q["options"]))
            question_text = f"{question_text}\n\nOptions:\n{opts}"

        reasoning, boxed = extract_boxed_and_reasoning(r["response"])
        # Drop anything where the helper failed to find a real \boxed{...}
        if not boxed or boxed == "\\" or "{" not in boxed:
            skipped += 1
            continue
        msgs = make_messages(system, question_text, reasoning, boxed)
        public_texts.append(apply_chat_template(msgs))

print(f"Public correct formatted: {len(public_texts)}  (skipped {skipped})")

In [ ]:
# Cell 8 — Combine + build HuggingFace Dataset
from datasets import Dataset

all_texts = numinamath_texts + public_texts
random.shuffle(all_texts)
print(f"Total training examples: {len(all_texts)}")
print(f"  NuminaMath: {len(numinamath_texts)}")
print(f"  Public correct: {len(public_texts)}")

# Sanity check token lengths
lengths = [len(tokenizer.encode(t)) for t in all_texts[:500]]
import statistics
print(f"Token length (sample 500) — mean: {statistics.mean(lengths):.0f}, "
      f"p95: {sorted(lengths)[int(0.95*len(lengths))]}, max: {max(lengths)}")

# Filter examples that exceed MAX_SEQ_LENGTH
all_texts_filtered = [t for t in all_texts if len(tokenizer.encode(t)) <= MAX_SEQ_LENGTH]
print(f"After length filter: {len(all_texts_filtered)} "
      f"({len(all_texts)-len(all_texts_filtered)} dropped)")

train_dataset = Dataset.from_dict({"text": all_texts_filtered})
print(train_dataset)

In [ ]:
# Cell 8b — Format sanity check (MUST READ before training)
# Verifies that the training string format matches what vLLM will produce at inference time.
# If these don't look like a natural continuation of each other, fix make_messages() BEFORE training.

print("=== Rendered training example (first 1500 chars) ===")
print(train_dataset[0]["text"][:1500])
print()
print("=== Inference-time prompt (what vLLM sees at generation) ===")
inf_msgs = [
    {"role": "system", "content": SYSTEM_PROMPT_MATH},
    {"role": "user",   "content": "What is 2+2?"},
]
print(tokenizer.apply_chat_template(inf_msgs, tokenize=False, add_generation_prompt=True))

In [ ]:
# Cell 9 — Train
import shutil, os, gc, torch
from trl import SFTTrainer, SFTConfig

# Clean up any prior checkpoint dir so we start fresh
if os.path.exists("/content/exp008_checkpoints"):
    shutil.rmtree("/content/exp008_checkpoints")

# Free any leftover VRAM from a previous failed trainer.train() before retrying
gc.collect()
torch.cuda.empty_cache()

# Hold out 200 examples for eval — catches format bugs that the sanity check misses
eval_size = min(200, len(train_dataset) // 10)
split = train_dataset.train_test_split(test_size=eval_size, seed=RANDOM_SEED)
train_split = split["train"]
eval_split  = split["test"]
print(f"Train: {len(train_split)}  Eval: {len(eval_split)}")

training_args = SFTConfig(
    output_dir="/content/exp008_checkpoints",
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    optim="adamw_8bit",
    bf16=True,                      # A100 always supports bf16 — no need for the conditional
    logging_steps=50,
    eval_strategy="steps",
    eval_steps=50,
    per_device_eval_batch_size=4,   # eval also OOMs at 8 since no grad checkpointing
    save_strategy="epoch",
    save_total_limit=1,             # keep only the last checkpoint to avoid filling /content
    seed=RANDOM_SEED,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    packing=True,                   # packs multiple short examples per window — ~3-4x fewer steps
    dataloader_num_workers=4,       # parallel data loading
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_split,
    eval_dataset=eval_split,
    args=training_args,
)

print("Starting training...")
trainer.train()
print("Training complete.")

In [ ]:
# Cell 10 — Merge adapter into base model + save as float16
#
# This dequantizes the 4-bit base and adds the LoRA deltas,
# producing a standard float16 model that vLLM can load directly.

MERGED_PATH = "/content/exp008_merged"

print("Merging adapter into base weights (float16)...")
model.save_pretrained_merged(
    MERGED_PATH,
    tokenizer,
    save_method="merged_16bit",
)
print(f"Merged model saved to {MERGED_PATH}")
!du -sh {MERGED_PATH}

In [ ]:
# Cell 11 — Push merged model to HuggingFace (private repo)
from huggingface_hub import HfApi

api = HfApi(token=HF_TOKEN)

# Create private repo if it doesn't exist
try:
    api.create_repo(HF_REPO_ID, private=True, exist_ok=True)
    print(f"Repo ready: https://huggingface.co/{HF_REPO_ID}")
except Exception as e:
    print(f"Repo creation note: {e}")

print("Uploading merged model (this will take a few minutes for ~8GB)...")
api.upload_folder(
    folder_path=MERGED_PATH,
    repo_id=HF_REPO_ID,
    repo_type="model",
    token=HF_TOKEN,
)
print(f"Done! Model at: https://huggingface.co/{HF_REPO_ID}")
print(f"Set HF_REPO_ID = '{HF_REPO_ID}' in your Kaggle inference notebook.")